In [1]:
import os

In [2]:
pwd("../")

'f:\\Condition2Cure\\notebook'

In [3]:
os.chdir("../")

In [4]:
pwd("../")

'f:\\Condition2Cure'

In [5]:
from pathlib import Path
from dataclasses import dataclass

In [6]:
from Condition2Cure.utils.helpers import *
from Condition2Cure.constants import *

In [7]:
@dataclass(frozen=True)
class FeatureEngineeringConfig:
    root_dir: Path
    cleaned_data_path: Path
    vectorizer_path: Path
    features_path: Path
    label_encoder_path: Path
    target_column: str
    max_features: int
    ngram_range: list

In [8]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_feature_engineering_config(self) -> FeatureEngineeringConfig:
        config = self.config.feature_engineering
        params = self.params.vectorizer
        schema = self.schema.target_column

        create_directories([config.root_dir])

        feature_engineering_config =  FeatureEngineeringConfig(
            root_dir=config.root_dir,
            cleaned_data_path=config.cleaned_data_path,
            vectorizer_path=config.vectorizer_path,
            features_path=config.features_path,
            label_encoder_path=config.label_encoder_path,
            target_column=schema.name,
            max_features=params.max_features,
            ngram_range=params.ngram_range
        )

        return feature_engineering_config


In [9]:
import os
import pandas as pd
import numpy as np
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from Condition2Cure import logger

class FeatureEngineering:
    def __init__(self, config: FeatureEngineeringConfig):
        self.config = config

    def transfom(self):
        logger.info("Loading cleaned data...")
        df = pd.read_csv(self.config.cleaned_data_path)

        if 'clean_review' not in df.columns or self.config.target_column not in df.columns:
            raise ValueError("Required columns missing in cleaned data.")

        df['clean_review'] = df['clean_review'].fillna("")


        logger.info("Fitting TF-IDF vectorizer...")
        vectorizer = TfidfVectorizer(
            max_features=self.config.max_features,
            ngram_range=tuple(self.config.ngram_range)
        )
        X = vectorizer.fit_transform(df['clean_review'])

        logger.info("Encoding target labels...")
        label_encoder = LabelEncoder()
        y = label_encoder.fit_transform(df[self.config.target_column])

        logger.info(f"TF-IDF features shape: {X.shape}")
        logger.info(f"Number of classes: {len(np.unique(y))}")

        logger.info("Saving vectorizer and label encoder...")
        os.makedirs(os.path.dirname(self.config.vectorizer_path), exist_ok=True)
        joblib.dump(vectorizer, self.config.vectorizer_path)
        joblib.dump(label_encoder, self.config.label_encoder_path)

        logger.info("Saving feature matrix and labels...")
        os.makedirs(os.path.dirname(self.config.features_path), exist_ok=True)
        np.save(self.config.features_path, {'X': X, 'y': y})

        logger.info("Feature engineering completed.")

In [10]:
try:
    config = ConfigurationManager()
    feature_engineering_config = config.get_feature_engineering_config()
    feature_engineering = FeatureEngineering(config=feature_engineering_config)
    feature_engineering.transfom()
except FileNotFoundError as e:
    logger.error(f"File not found: {e}")
except KeyError as e:
    logger.error(f"Missing key in configuration: {str(e)}")
except Exception as e:
    logger.error(f"Unexpected error: {str(e)}") 

[2025-05-21 12:07:16,693: INFO: helpers: yaml file: config\config.yaml loaded successfully]
[2025-05-21 12:07:16,699: INFO: helpers: yaml file: config\params.yaml loaded successfully]


[2025-05-21 12:07:16,706: INFO: helpers: yaml file: config\schema.yaml loaded successfully]
[2025-05-21 12:07:16,708: INFO: helpers: created directory at: artifacts]
[2025-05-21 12:07:16,710: INFO: helpers: created directory at: artifacts/feature_engineering]
[2025-05-21 12:07:16,710: INFO: 1510937143: Loading cleaned data...]
[2025-05-21 12:07:17,519: INFO: 1510937143: Fitting TF-IDF vectorizer...]
[2025-05-21 12:07:32,503: INFO: 1510937143: Encoding target labels...]
[2025-05-21 12:07:32,527: INFO: 1510937143: TF-IDF features shape: (60369, 10000)]
[2025-05-21 12:07:32,533: INFO: 1510937143: Number of classes: 7]
[2025-05-21 12:07:32,535: INFO: 1510937143: Saving vectorizer and label encoder...]
[2025-05-21 12:07:32,886: INFO: 1510937143: Saving feature matrix and labels...]
[2025-05-21 12:07:32,939: INFO: 1510937143: Feature engineering completed.]
